# Anotador TDAH · 02/04 · Backend `langchain`

Usa `ChatOllama` de LangChain con `with_structured_output(method="json_schema")`: el esquema JSON de la anotación se pasa a Ollama como formato obligatorio, y el modelo queda **forzado a nivel de decodificación** a emitir un JSON que cumpla el esquema. No depende de que el modelo sepa hacer tool calling.

Lo esperable: fallo de formato ≈ 0. Comprobar si al forzar el esquema también estabiliza el *contenido* (los ítems elegidos) o solo la forma.


## 1 · Parámetros

In [1]:
import datetime as dt
import json
import sqlite3
import time

import pandas as pd

SEMANA       = 1            # semana de seguimiento (el dataset llega a la 24)
PACIENTES    = None         # None = todos los de la semana; o lista: ["P001", "P003"]
REPETICIONES = 3            # veces que se anota cada entrada
TEMPERATURA  = 0.7
MODELO       = "gemma4:e4b" 
# --- Rutas y conexión ---
RUTA_BD     = "datos/anotador.db"
OLLAMA_URL  = "http://127.0.0.1:11002"  
INSTRUMENTO = "instrumentos/brief2.json"

BACKEND     = "langchain"
EXPERIMENTO = f"{BACKEND}-s{SEMANA}-t{TEMPERATURA}-{dt.date.today():%Y%m%d}"
print(f"Código de experimento: {EXPERIMENTO}")

Código de experimento: langchain-s1-t0.7-20260713


## 2 · Datos

Las entradas (texto libre de los padres) de la semana elegida, con el contexto del paciente (edad calculada a la fecha de la observación, sexo, quién informa).

In [2]:
instrumento = json.load(open(INSTRUMENTO, encoding="utf-8"))
print(f"Instrumento: {instrumento['nombre']} ({len(instrumento['items'])} ítems)")

con = sqlite3.connect(RUTA_BD)

entradas = pd.read_sql(
    '''
    SELECT e.id_entrada, e.id_paciente, c.rol AS informante, e.fecha,
           p.fecha_nacimiento, p.sexo, e.texto
    FROM entrada e
    JOIN paciente p USING (id_paciente)
    JOIN cuidador c ON c.id_cuidador = e.id_cuidador
    JOIN referencia_sintetica r USING (id_entrada)
    WHERE r.semana = ?
    ORDER BY e.id_paciente
    ''',
    con, params=[SEMANA],
)
if PACIENTES:
    entradas = entradas[entradas["id_paciente"].isin(PACIENTES)]


def calcular_edad(nacimiento, observacion):
    nac = pd.to_datetime(nacimiento).date()
    obs = pd.to_datetime(observacion).date()
    return obs.year - nac.year - ((obs.month, obs.day) < (nac.month, nac.day))


entradas["edad"] = [
    calcular_edad(n, f) for n, f in zip(entradas["fecha_nacimiento"], entradas["fecha"])
]

print(f"Semana {SEMANA}: {len(entradas)} entradas de {entradas['id_paciente'].nunique()} pacientes")
entradas[["id_entrada", "id_paciente", "informante", "edad", "sexo", "texto"]].head()

Instrumento: BRIEF-2 Familia (63 ítems)
Semana 1: 30 entradas de 30 pacientes


,id_entrada,id_paciente,informante,edad,sexo,texto
0,1,PAC001,madre,7,masculino,"Hoy ha sido un día horrible, la verdad. Marco ..."
1,25,PAC002,madre,11,femenino,"Hola, soy la madre de Lucía. Nos dijeron que t..."
2,49,PAC003,padre,15,masculino,Soy el padre de Alejandro. La psiquiatra nos h...
3,73,PAC004,madre,6,femenino,"Somos los padres de Sofía, tiene 6 años. Esta ..."
4,97,PAC005,madre,8,masculino,Soy la madre de Diego. Diego vive conmigo de l...


## 3 · Prompts

El prompt de sistema se construye desde el instrumento (`brief2.json`): catálogo de ítems, escalas y niveles de alerta. El de usuario lleva el contexto del paciente y su texto.

In [4]:
COMILLAS = '"' * 3  


def construir_prompt_sistema(instrumento):
    catalogo = "\n".join(
        f"  {it['id']}: [{it['escala']}] {it['texto']}" for it in instrumento["items"]
    )
    escalas = "\n".join(f"  - {e}: {d}" for e, d in instrumento["escalas"].items())
    n = instrumento["niveles_alerta"]
    return f'''Eres un {instrumento["rol_anotador"]}.

Tu tarea es analizar el texto libre de observación de un padre/madre sobre su hijo/a
y producir una anotación clínica estructurada en formato JSON, basada en el instrumento
{instrumento["nombre"]}.

## CATÁLOGO DE ÍTEMS ({len(instrumento["items"])} ítems)
{catalogo}

## ESCALAS
{escalas}

## NIVELES DE ALERTA
{" | ".join(n)}

## INSTRUCCIONES DE SALIDA
Responde ÚNICAMENTE con un objeto JSON válido, sin texto antes ni después, sin markdown.
Estructura requerida:
{{
  "items_detectados": [lista de números de ítem observables en el texto],
  "escalas_afectadas": [lista de escalas correspondientes],
  "nivel_alerta": "{n[0]}|{n[1]}|{n[2]}",
  "nota_clinica": "resumen clínico de 1-3 frases para el médico",
  "justificacion": "explicación del razonamiento (para auditoría)"
}}'''


def construir_prompt_usuario(e):
    return f'''## CONTEXTO DEL PACIENTE
- Edad: {e.edad} años
- Sexo: {e.sexo}
- Informante: {e.informante}

## TEXTO DEL PADRE/MADRE
{COMILLAS}{e.texto}{COMILLAS}

Analiza el texto y genera el JSON de anotación clínica.'''


prompt_sistema = construir_prompt_sistema(instrumento)
print(prompt_sistema[:400] + "\n[...]")

Eres un psicólogo clínico infantil especializado en TDAH y en el instrumento BRIEF-2.

Tu tarea es analizar el texto libre de observación de un padre/madre sobre su hijo/a
y producir una anotación clínica estructurada en formato JSON, basada en el instrumento
BRIEF-2 Familia.

## CATÁLOGO DE ÍTEMS (63 ítems)
  1: [inhibicion] Es inquieto o inquieta.
  2: [flexibilidad] Se resiste o le cuesta acept
[...]


## 4 · Backend Langchain

In [5]:
from pydantic import BaseModel, Field


class Anotacion(BaseModel):
    items_detectados: list[int] = Field(default_factory=list)
    escalas_afectadas: list[str] = Field(default_factory=list)
    nivel_alerta: str = "bajo"
    nota_clinica: str = ""
    justificacion: str = ""

In [7]:
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_ollama import ChatOllama

llm = ChatOllama(
    model=MODELO,
    base_url=OLLAMA_URL,
    temperature=TEMPERATURA,
    num_predict=2048,
)
llm_estructurado = llm.with_structured_output(Anotacion, method="json_schema")


def anotar(prompt_sistema, prompt_usuario):
    '''Llama al modelo y devuelve (anotacion | None, respuesta_cruda).'''
    try:
        obj = llm_estructurado.invoke(
            [SystemMessage(prompt_sistema), HumanMessage(prompt_usuario)]
        )
    except Exception as e:
        return None, f"error: {e}"
    if obj is None:
        return None, ""
    datos = obj if isinstance(obj, dict) else obj.model_dump()
    return datos, json.dumps(datos, ensure_ascii=False)

## 5 · Una anotación de ejemplo

Antes de lanzar el experimento completo, una sola entrada para ver la anotación final que produce este backend.

In [8]:
ejemplo = entradas.iloc[0]
print(f"Paciente {ejemplo.id_paciente} · {ejemplo.informante} · semana {SEMANA}")
print(f"Texto: {ejemplo.texto[:200]}...\n")

t0 = time.time()
anotacion, cruda = anotar(prompt_sistema, construir_prompt_usuario(ejemplo))
print(f"Latencia: {time.time() - t0:.1f}s\n")

if anotacion is None:
    print("[FALLO DE FORMATO] El modelo no devolvió un JSON válido:")
    print(cruda[:500])
else:
    print("ANOTACIÓN FINAL:")
    print(json.dumps(anotacion, indent=2, ensure_ascii=False))

Paciente PAC001 · madre · semana 1
Texto: Hoy ha sido un día horrible, la verdad. Marco lleva tres semanas desde el diagnóstico y yo sigo sin saber muy bien cómo manejarlo. Esta mañana no había manera de que se sentara a desayunar, estaba sal...

Latencia: 17.9s

ANOTACIÓN FINAL:
{
  "items_detectados": [
    1,
    3,
    4,
    10
  ],
  "escalas_afectadas": [
    "inhibicion",
    "memoria_trabajo",
    "supervision_conducta",
    "control_impulsos"
  ],
  "nivel_alerta": "moderado",
  "nota_clinica": "Se observa una marcada dificultad en el control de impulsos y la hiperactividad motora (saltar de la silla), así como problemas significativos con la memoria de trabajo al seguir instrucciones complejas. La madre reporta también dificultades para gestionar las consecuencias sociales de su conducta.",
  "justificacion": "1 (inhibicion): 'estaba saltando de la silla literalmente cada dos por tres' (Hiperactividad). \n3 (memoria_trabajo): 'Le digo que tiene que hacer tres cosas... y siem

## 6. Experimento Backend Langchain

### 6.1. Repetir Experimento 
Mismo día, mismo código (borrar y repetir)

In [ ]:
# Borra todas las filas de este experimento langchain y empieza de cero
con.execute("DELETE FROM experimento WHERE codigo = ?", [EXPERIMENTO])
con.commit()
print(f"Borradas filas de '{EXPERIMENTO}'. Listo para relanzar.")

### 6.2. Añadir Experimento mismo dia
Mismo día, código nuevo (conservar el anterior)

In [ ]:
EXPERIMENTO = f"{BACKEND}-s{SEMANA}-t{TEMPERATURA}-{dt.datetime.now():%Y%m%d-%H%M}"

## 6 · Experimento

Anota cada entrada de la semana `REPETICIONES` veces y guarda cada resultado en la tabla `experimento` con el código `EXPERIMENTO`. 

In [9]:
con.execute('''
CREATE TABLE IF NOT EXISTS experimento (
    id                INTEGER PRIMARY KEY,
    codigo            TEXT NOT NULL,      -- código del experimento (para comparar)
    creada_en         TEXT NOT NULL,
    backend           TEXT NOT NULL,
    modelo            TEXT NOT NULL,
    temperatura       REAL NOT NULL,
    semana            INTEGER,
    id_paciente       TEXT,
    id_entrada        INTEGER,
    repeticion        INTEGER,
    formato_ok        INTEGER,
    items_detectados  TEXT,               -- JSON: [int]
    escalas_afectadas TEXT,               -- JSON: [str]
    nivel_alerta      TEXT,
    nota_clinica      TEXT,
    justificacion     TEXT,
    latencia_s        REAL
)''')
con.commit()

total = len(entradas) * REPETICIONES
print(f"Experimento '{EXPERIMENTO}': {len(entradas)} entradas × {REPETICIONES} repeticiones "
      f"= {total} llamadas al modelo")

hechas = 0
for _, e in entradas.iterrows():
    prompt_usuario = construir_prompt_usuario(e)
    for rep in range(REPETICIONES):
        t0 = time.time()
        anotacion, cruda = anotar(prompt_sistema, prompt_usuario)
        latencia = time.time() - t0
        ok = anotacion is not None
        a = anotacion or {}
        con.execute(
            "INSERT INTO experimento (codigo, creada_en, backend, modelo, temperatura, "
            "semana, id_paciente, id_entrada, repeticion, formato_ok, items_detectados, "
            "escalas_afectadas, nivel_alerta, nota_clinica, justificacion, latencia_s) "
            "VALUES (?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?)",
            (EXPERIMENTO, dt.datetime.now().isoformat(timespec="seconds"), BACKEND,
             MODELO, TEMPERATURA, SEMANA, e.id_paciente, int(e.id_entrada), rep,
             int(ok), json.dumps(a.get("items_detectados", [])),
             json.dumps(a.get("escalas_afectadas", [])), a.get("nivel_alerta"),
             a.get("nota_clinica"), a.get("justificacion"), latencia),
        )
        con.commit()
        hechas += 1
        estado = "ok" if ok else "FALLO DE FORMATO"
        print(f"  [{hechas:>3}/{total}] {e.id_paciente} rep {rep + 1} → {estado} ({latencia:.1f}s)")

print(f"\nGuardado en la tabla `experimento` con codigo = '{EXPERIMENTO}'")

Experimento 'langchain-s1-t0.7-20260713': 30 entradas × 3 repeticiones = 90 llamadas al modelo
  [  1/90] PAC001 rep 1 → ok (18.0s)
  [  2/90] PAC001 rep 2 → ok (11.3s)
  [  3/90] PAC001 rep 3 → ok (9.9s)
  [  4/90] PAC002 rep 1 → ok (9.6s)
  [  5/90] PAC002 rep 2 → ok (8.6s)
  [  6/90] PAC002 rep 3 → ok (9.2s)
  [  7/90] PAC003 rep 1 → ok (10.2s)
  [  8/90] PAC003 rep 2 → ok (9.4s)
  [  9/90] PAC003 rep 3 → ok (8.8s)
  [ 10/90] PAC004 rep 1 → ok (10.5s)
  [ 11/90] PAC004 rep 2 → ok (9.9s)
  [ 12/90] PAC004 rep 3 → ok (10.6s)
  [ 13/90] PAC005 rep 1 → ok (9.9s)
  [ 14/90] PAC005 rep 2 → ok (9.8s)
  [ 15/90] PAC005 rep 3 → ok (10.1s)
  [ 16/90] PAC006 rep 1 → ok (8.0s)
  [ 17/90] PAC006 rep 2 → ok (8.1s)
  [ 18/90] PAC006 rep 3 → ok (7.7s)
  [ 19/90] PAC007 rep 1 → ok (9.1s)
  [ 20/90] PAC007 rep 2 → ok (7.8s)
  [ 21/90] PAC007 rep 3 → ok (9.7s)
  [ 22/90] PAC008 rep 1 → ok (8.4s)
  [ 23/90] PAC008 rep 2 → ok (9.3s)
  [ 24/90] PAC008 rep 3 → ok (8.5s)
  [ 25/90] PAC009 rep 1 → ok (7.0s)

## 7 · Resultados

- `formato_ok`: salidas que fueron JSON válido.
- `acuerdo_nivel` (0–1): repeticiones que coincide con el nivel de alerta más frecuente de ese paciente. 1.0 = el modelo dice siempre lo mismo.
- `latencia_media`: segundos por anotación.

In [22]:
df = pd.read_sql(
    "SELECT * FROM experimento WHERE codigo = ?", con, params=[EXPERIMENTO]
)
print(f"{len(df)} anotaciones del experimento '{EXPERIMENTO}'\n")


def acuerdo_modal(niveles):
    '''Fracción de repeticiones que coincide con el nivel más frecuente.'''
    s = niveles.dropna()
    return round(s.value_counts().iloc[0] / len(s), 2) if len(s) else None


resumen = df.groupby("id_paciente").agg(
    repeticiones=("repeticion", "count"),
    formato_ok=("formato_ok", "mean"),
    acuerdo_nivel=("nivel_alerta", acuerdo_modal),
    latencia_media=("latencia_s", "mean"),
).round(2)

print(f"Formato válido: {df['formato_ok'].mean():.0%}")
print(f"Acuerdo medio del nivel de alerta entre repeticiones: {resumen['acuerdo_nivel'].mean():.2f}")
print(f"Latencia media por anotación: {df['latencia_s'].mean():.1f}s\n")
resumen

90 anotaciones del experimento 'langchain-s1-t0.7-20260713'

Formato válido: 100%
Acuerdo medio del nivel de alerta entre repeticiones: 0.95
Latencia media por anotación: 8.7s



,repeticiones,formato_ok,acuerdo_nivel,latencia_media
id_paciente,,,,
PAC001,3,1.0,1.00,13.07
PAC002,3,1.0,1.00,9.13
PAC003,3,1.0,1.00,9.47
PAC004,3,1.0,1.00,10.33
PAC005,3,1.0,1.00,9.91
PAC006,3,1.0,1.00,7.93
PAC007,3,1.0,1.00,8.89
PAC008,3,1.0,1.00,8.73
PAC009,3,1.0,1.00,7.92


In [23]:
# Las anotaciones de un paciente concreto, repetición a repetición
UN_PACIENTE = df["id_paciente"].iloc[0]   # cambiar por el que interese

detalle = df[df["id_paciente"] == UN_PACIENTE]
for _, fila in detalle.iterrows():
    print(f"— repetición {fila.repeticion}: nivel={fila.nivel_alerta} "
          f"items={fila.items_detectados}")
    print(f"  nota: {fila.nota_clinica}\n")

— repetición 0: nivel=moderado items=[1, 3, 4, 10]
  nota: Se observa un patrón de hiperactividad motora (saltar de la silla) e impulsividad conductual (empujar a otro niño). Además, presenta dificultades significativas en la memoria de trabajo y la conciencia del impacto social de sus acciones. Estas áreas requieren apoyo terapéutico específico.

— repetición 1: nivel=moderado items=[1, 3, 4, 10]
  nota: El paciente presenta dificultades significativas en la regulación motora y el control de impulsos (saltar, empujar), además de problemas para retener información secuencial durante las tareas escolares. Estos síntomas sugieren un impacto considerable en su funcionamiento diario que requiere intervención.

— repetición 2: nivel=moderado items=[1, 4, 3]
  nota: El paciente muestra una hiperactividad motora significativa (saltar de la silla) e impulsividad evidente en su interacción social. Además, presenta dificultades notables con la memoria operativa y la atención secuencial al realiz

## 8 · Siguiente paso

Comparar este experimento con los de los otros backends (u otros parámetros) en `05_comparacion_experimentos.ipynb`, usando los códigos de experimento.